# Leave-one-cluster-out validation

Evaluate the monoterpene synthase classifier using the sequence, structural,
and active-site feature clusters supplied with the repository. Each cluster
is withheld once while the model is trained on all remaining proteins.

The saved split manifests make this analysis independent of the sequence
and structural clustering programs. The stratified random-fold manifest
is checked separately because it belongs to the nearest-neighbour
comparison, not to the reported XGBoost LOCO results.


## Load the modelling dataset


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from xgboost import XGBClassifier


def repository_root(start):
    for directory in (start, *start.parents):
        if (directory / "FullDataset.csv").is_file():
            return directory
    raise FileNotFoundError("Run this notebook from within the ATC repository.")


ROOT = repository_root(Path.cwd().resolve())
LOCO_DIRECTORY = ROOT / "SupplementaryInformation" / "LOCO"
manifest_override = os.environ.get("ATC_LOCO_MANIFEST_DIR")
SPLIT_DIRECTORY = (
    Path(manifest_override).resolve() if manifest_override else LOCO_DIRECTORY
)

IDENTIFIER = "Protein"
TARGET = "Cyclical"
FEATURES = [
    "Alanine - A",
    "Methionine - M",
    "Tryptophan - W",
    "Isoleucine - I",
    "Cysteine - C",
    "Histidine - H",
    "Asparagine - N",
    "Positive charge",
    "Negative charge",
    "Polar",
    "Non-polar",
    "Amino Acid based volume Score",
]

dataset = pd.read_csv(ROOT / "FullDataset.csv")
missing_columns = sorted({IDENTIFIER, TARGET, *FEATURES} - set(dataset.columns))
if missing_columns:
    raise ValueError(f"FullDataset.csv is missing columns: {missing_columns}")

dataset[IDENTIFIER] = dataset[IDENTIFIER].astype(str).str.strip()
duplicate_ids = dataset.loc[dataset[IDENTIFIER].duplicated(), IDENTIFIER].tolist()
if duplicate_ids:
    raise ValueError(f"FullDataset.csv repeats protein identifiers: {duplicate_ids}")

dataset[TARGET] = pd.to_numeric(dataset[TARGET], errors="raise").astype(int)
dataset[FEATURES] = dataset[FEATURES].apply(pd.to_numeric, errors="raise")
if dataset[FEATURES].isna().any().any():
    raise ValueError("The model features contain missing values.")
if set(dataset[TARGET]) != {0, 1}:
    raise ValueError("Cyclical must contain both classes: 0 and 1.")

dataset = dataset.set_index(IDENTIFIER, drop=False)
print(f"Modelling proteins: {len(dataset)}")
print(f"Linear: {(dataset[TARGET] == 0).sum()}")
print(f"Cyclic: {(dataset[TARGET] == 1).sum()}")


## Validate the saved folds

Sequence manifests contain explicit training and test assignments.
Structural manifests may contain test assignments only; their training
proteins are recovered as the remaining modelling proteins.


In [ ]:
def load_folds(filename):
    path = SPLIT_DIRECTORY / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing split manifest: {path}")

    manifest = pd.read_csv(path, dtype=str)
    required = {"Protein_ID", "Fold", "Split"}
    missing = sorted(required - set(manifest.columns))
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")

    for column in required:
        if manifest[column].isna().any():
            raise ValueError(f"{path.name} contains missing {column} values.")
        manifest[column] = manifest[column].str.strip()
    manifest["Split"] = manifest["Split"].str.casefold()

    invalid_labels = sorted(set(manifest["Split"]) - {"train", "test"})
    if invalid_labels:
        raise ValueError(f"{path.name} has invalid split labels: {invalid_labels}")

    dataset_ids = set(dataset.index)
    unknown = sorted(set(manifest["Protein_ID"]) - dataset_ids)
    if unknown:
        raise ValueError(f"{path.name} contains unknown proteins: {unknown}")

    folds = []
    held_out = []
    for name, assignments in manifest.groupby("Fold", sort=True):
        test = assignments.loc[
            assignments["Split"].eq("test"), "Protein_ID"
        ].tolist()
        train = assignments.loc[
            assignments["Split"].eq("train"), "Protein_ID"
        ].tolist()
        if not test:
            raise ValueError(f"{path.name} {name} has no test proteins.")
        if len(test) != len(set(test)) or len(train) != len(set(train)):
            raise ValueError(f"{path.name} {name} repeats protein assignments.")

        test_ids = set(test)
        if not train:
            train = [protein for protein in dataset.index if protein not in test_ids]

        if set(train) & test_ids:
            raise ValueError(f"{path.name} {name} overlaps training and test sets.")
        if set(train) | test_ids != dataset_ids:
            raise ValueError(f"{path.name} {name} does not include every protein.")

        held_out.extend(test)
        folds.append({"fold": name, "train": train, "test": test})

    repeated = sorted(pd.Series(held_out)[pd.Series(held_out).duplicated()].unique())
    missing_test = sorted(dataset_ids - set(held_out))
    if repeated or missing_test:
        raise ValueError(
            f"{path.name} must withhold each protein once; "
            f"repeated={repeated}, missing={missing_test}."
        )

    print(f"{path.name}: {len(folds)} folds, {len(held_out)} test proteins")
    return folds


protocols = {
    "Sequence LOCO": load_folds("splits_seq_loco.csv"),
    "Structure LOCO": load_folds("splits_struct_loco.csv"),
}

feature_manifest = SPLIT_DIRECTORY / "splits_feature_loco.csv"
if feature_manifest.is_file():
    protocols["All-feature LOCO"] = load_folds(feature_manifest.name)
else:
    print("All-feature LOCO requires splits_feature_loco.csv.")

random_manifest = SPLIT_DIRECTORY / "splits_random_k5.csv"
if random_manifest.is_file():
    try:
        load_folds(random_manifest.name)
    except ValueError as error:
        print(f"Random-fold manifest is invalid: {error}")


## Train and evaluate the classifier

Each fold uses the final 12-feature XGBoost model. Sample weights are
calculated from the class frequencies of that fold's training proteins.
Predictions from all held-out clusters are combined before calculating
classification metrics.


In [ ]:
def build_model():
    return XGBClassifier(
        eval_metric="logloss",
        reg_alpha=0.1,
        reg_lambda=1.0,
        verbosity=0,
        random_state=42,
        tree_method="hist",
        learning_rate=0.1,
    )


def classification_metrics(observed, predicted):
    observed = np.asarray(observed, dtype=int)
    predicted = np.asarray(predicted, dtype=int)
    metrics = {
        "Observations": len(observed),
        "Accuracy": accuracy_score(observed, predicted),
        "Balanced accuracy": balanced_accuracy_score(observed, predicted),
        "MCC": matthews_corrcoef(observed, predicted),
    }
    for label, name in ((0, "linear"), (1, "cyclic")):
        metrics[f"Precision ({name})"] = precision_score(
            observed, predicted, pos_label=label, zero_division=0
        )
        metrics[f"Recall ({name})"] = recall_score(
            observed, predicted, pos_label=label, zero_division=0
        )
        metrics[f"F1 ({name})"] = f1_score(
            observed, predicted, pos_label=label, zero_division=0
        )
    return metrics


def evaluate_protocol(protocol, folds):
    records = []
    for fold in folds:
        training = dataset.loc[fold["train"]]
        testing = dataset.loc[fold["test"]]
        labels = training[TARGET]
        if set(labels) != {0, 1}:
            raise ValueError(
                f"{protocol} {fold['fold']} needs both training classes."
            )

        frequencies = labels.value_counts(normalize=True)
        weights = labels.map(1 / frequencies).to_numpy()
        model = build_model().fit(
            training[FEATURES], labels, sample_weight=weights
        )
        predictions = model.predict(testing[FEATURES]).astype(int)
        probabilities = model.predict_proba(testing[FEATURES])[:, 1]

        records.extend(
            {
                "Protocol": protocol,
                "Fold": fold["fold"],
                "Protein": protein,
                "Observed": int(observed),
                "Predicted": int(predicted),
                "Cyclic probability": float(probability),
            }
            for protein, observed, predicted, probability in zip(
                testing.index, testing[TARGET], predictions, probabilities
            )
        )

    predictions = pd.DataFrame(records)
    scores = classification_metrics(
        predictions["Observed"], predictions["Predicted"]
    )
    matrix = confusion_matrix(
        predictions["Observed"], predictions["Predicted"], labels=[0, 1]
    )
    print(f"\n{protocol}")
    print(f"Confusion matrix [[TN, FP], [FN, TP]]:\n{matrix}")
    return predictions, {"Protocol": protocol, "Folds": len(folds), **scores}


prediction_tables = []
results = []
for protocol, folds in protocols.items():
    predictions, scores = evaluate_protocol(protocol, folds)
    prediction_tables.append(predictions)
    results.append(scores)

results = pd.DataFrame(results)
predictions = pd.concat(prediction_tables, ignore_index=True)
output_directory = LOCO_DIRECTORY / "validation_outputs"
output_directory.mkdir(exist_ok=True)
results.to_csv(output_directory / "loco_metrics.csv", index=False)
predictions.to_csv(output_directory / "loco_predictions.csv", index=False)

print("\nLOCO classification results")
print(results.round(4).to_string(index=False))
